<a href="https://colab.research.google.com/github/dghiberdic/4cblw010-g9/blob/main/PET_FTIR_CNN_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU available: []


In [2]:
from google.colab import files

uploaded = files.upload()

Saving y_labels.npy to y_labels.npy


In [3]:
from google.colab import files

uploaded = files.upload()

Saving X_spectra.npy to X_spectra.npy


In [4]:
import os

print(os.listdir())

['.config', 'X_spectra.npy', 'y_labels.npy', 'sample_data']


In [6]:
if X.ndim == 2:
    X = X[..., np.newaxis]

print("CNN input shape:", X.shape)

CNN input shape: (200, 3600, 1)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (160, 3600, 1)
X_test: (40, 3600, 1)
y_train: (160, 20)
y_test: (40, 20)


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_ftir_cnn(input_length, num_labels):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),

        layers.Conv1D(32, kernel_size=7, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.GlobalAveragePooling1D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(num_labels, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model


input_length = X_train.shape[1]
num_labels = y_train.shape[1]

model = build_ftir_cnn(input_length, num_labels)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 3600, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1800, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1800, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 900, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 900, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 450, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,356 (212.33 KB)

 Trainable params: 54,356 (212.33 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 219ms/step - auc: 0.5065 - binary_accuracy: 0.5023 - loss: 0.6930 - val_auc: 0.4653 - val_binary_accuracy: 0.4781 - val_loss: 0.6958
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 168ms/step - auc: 0.5404 - binary_accuracy: 0.5234 - loss: 0.6908 - val_auc: 0.4687 - val_binary_accuracy: 0.4844 - val_loss: 0.6977
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - auc: 0.5526 - binary_accuracy: 0.5398 - loss: 0.6891 - val_auc: 0.4714 - val_binary_accuracy: 0.4844 - val_loss: 0.7008
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - auc: 0.5471 - binary_accuracy: 0.5285 - loss: 0.6894 - val_auc: 0.4687 - val_binary_accuracy: 0.4906 - val_loss: 0.7024
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 165ms/step - auc: 0.5562 - binary_accuracy: 0.5379 - loss: 0.6881 - val_auc: 0.4730 - val_binary_accuracy: 0.4844 - val_loss: 0.7034
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 174ms/step - auc: 0.5522 - binary_accuracy: 0.5383 - loss: 0.6885 - val_auc: 0.4728 - val_binary_accu

In [10]:
from sklearn.metrics import f1_score, classification_report

LABEL_NAMES = [
    "ester",
    "carboxylic_acid",
    "alkane",
    "alkene",
    "alcohol",
    "arene",
    "amine",
    "ketone",
    "ether",
    "imine",
    "sulfonamide",
    "acyl_halide",
    "phosphate",
    "aldehyde",
    "nitro",
    "enamine",
    "azo",
    "sulfonic_acid",
    "amide",
    "peroxide",
]

y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int)

micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
samples_f1 = f1_score(y_test, y_pred, average="samples", zero_division=0)

print("CNN Evaluation")
print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)
print("Samples F1:", samples_f1)

print(classification_report(
    y_test,
    y_pred,
    target_names=LABEL_NAMES,
    zero_division=0
))

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 455ms/step
CNN Evaluation
Micro F1: 0.4196185286103542
Macro F1: 0.25858320243230026
Samples F1: 0.4105226904401591
                 precision    recall  f1-score   support

          ester       0.00      0.00      0.00        22
carboxylic_acid       0.40      1.00      0.57        16
         alkane       0.47      1.00      0.64        19
         alkene       0.47      1.00      0.64        19
        alcohol       0.00      0.00      0.00        19
          arene       0.00      0.00      0.00        24
          amine       0.00      0.00      0.00        23
         ketone       0.55      1.00      0.71        22
          ether       0.00      0.00      0.00        25
          imine       0.55      1.00      0.71        22
    sulfonamide       0.00      0.00      0.00        21
    acyl_halide       0.57      1.00      0.73        23
      phosphate       0.00      0.00      0.00        23
       aldehyde       0.00      0.00      0.00        20


In [1]:
from google.colab import files

uploaded = files.upload()

Saving X_spectra.npy to X_spectra.npy
Saving y_labels.npy to y_labels.npy


In [2]:
import numpy as np
import os

print(os.listdir())

X = np.load("X_spectra.npy")
y = np.load("y_labels.npy")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X min:", X.min())
print("X max:", X.max())
print("y unique:", np.unique(y))

['.config', 'X_spectra.npy', 'y_labels.npy', 'sample_data']
X shape: (13253, 1800)
y shape: (13253, 20)
X min: nan
X max: nan
y unique: [0 1]


In [3]:
import numpy as np

print("Total NaNs in X:", np.isnan(X).sum())
print("Rows with at least one NaN:", np.isnan(X).any(axis=1).sum())

Total NaNs in X: 52200
Rows with at least one NaN: 29


In [4]:
valid_rows = ~np.isnan(X).any(axis=1)

X_clean = X[valid_rows]
y_clean = y[valid_rows]

print("Original X:", X.shape)
print("Clean X:", X_clean.shape)
print("Original y:", y.shape)
print("Clean y:", y_clean.shape)

print("NaNs after cleaning:", np.isnan(X_clean).sum())
print("X clean min:", np.min(X_clean))
print("X clean max:", np.max(X_clean))

Original X: (13253, 1800)
Clean X: (13224, 1800)
Original y: (13253, 20)
Clean y: (13224, 20)
NaNs after cleaning: 0
X clean min: 0.0
X clean max: 1.0


In [5]:
import numpy as np

X = np.load("X_spectra.npy")
y = np.load("y_labels.npy")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X min:", X.min())
print("X max:", X.max())
print("y unique values:", np.unique(y))

X shape: (200, 3600)
y shape: (200, 20)
X min: 4.167095124074649e-06
X max: 0.9999998314950046
y unique values: [0 1]


In [5]:
X = X_clean
y = y_clean

if X.ndim == 2:
    X = X[..., np.newaxis]

print("CNN input shape:", X.shape)
print("y shape:", y.shape)

CNN input shape: (13224, 1800, 1)
y shape: (13224, 20)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (10579, 1800, 1)
X_test: (2645, 1800, 1)
y_train: (10579, 20)
y_test: (2645, 20)


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

LABEL_NAMES = [
    "ester",
    "carboxylic_acid",
    "alkane",
    "alkene",
    "alcohol",
    "arene",
    "amine",
    "ketone",
    "ether",
    "imine",
    "sulfonamide",
    "acyl_halide",
    "phosphate",
    "aldehyde",
    "nitro",
    "enamine",
    "azo",
    "sulfonic_acid",
    "amide",
    "peroxide",
]

def build_ftir_cnn(input_length, num_labels):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),

        layers.Conv1D(32, kernel_size=7, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        layers.MaxPooling1D(pool_size=2),

        layers.GlobalAveragePooling1D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),

        layers.Dense(num_labels, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

input_length = X_train.shape[1]
num_labels = y_train.shape[1]

model = build_ftir_cnn(input_length, num_labels)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1800, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 900, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 900, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 450, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 450, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 225, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,356 (212.33 KB)

 Trainable params: 54,356 (212.33 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 55s 185ms/step - auc: 0.8519 - binary_accuracy: 0.8882 - loss: 0.3062 - val_auc: 0.8922 - val_binary_accuracy: 0.8975 - val_loss: 0.2591
Epoch 2/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 51s 193ms/step - auc: 0.8883 - binary_accuracy: 0.8977 - loss: 0.2646 - val_auc: 0.8945 - val_binary_accuracy: 0.8978 - val_loss: 0.2567
Epoch 3/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 49s 186ms/step - auc: 0.8916 - binary_accuracy: 0.8979 - loss: 0.2607 - val_auc: 0.8961 - val_binary_accuracy: 0.8979 - val_loss: 0.2544
Epoch 4/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 48s 183ms/step - auc: 0.8938 - binary_accuracy: 0.8983 - loss: 0.2579 - val_auc: 0.8966 - val_binary_accuracy: 0.8978 - val_loss: 0.2550
Epoch 5/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 50s 188ms/step - auc: 0.8945 - binary_accuracy: 0.8986 - loss: 0.2569 - val_auc: 0.8979 - val_binary_accuracy: 0.8978 - val_loss: 0.2528
Epoch 6/40
265/265 ━━━━━━━━━━━━━━━━━━━━ 81s 183ms/step - auc: 0.8957 - binary_accuracy: 0.8985 - loss: 0.2556 - val

In [9]:
from sklearn.metrics import f1_score, classification_report

LABEL_NAMES = [
    "ester",
    "carboxylic_acid",
    "alkane",
    "alkene",
    "alcohol",
    "arene",
    "amine",
    "ketone",
    "ether",
    "imine",
    "sulfonamide",
    "acyl_halide",
    "phosphate",
    "aldehyde",
    "nitro",
    "enamine",
    "azo",
    "sulfonic_acid",
    "amide",
    "peroxide",
]

y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int)

micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
samples_f1 = f1_score(y_test, y_pred, average="samples", zero_division=0)

print("CNN Evaluation")
print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)
print("Samples F1:", samples_f1)

print(classification_report(
    y_test,
    y_pred,
    target_names=LABEL_NAMES,
    zero_division=0
))

83/83 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step
CNN Evaluation
Micro F1: 0.5796361840480698
Macro F1: 0.09207832273653282
Samples F1: 0.5740835849720538
                 precision    recall  f1-score   support

          ester       0.00      0.00      0.00       412
carboxylic_acid       0.00      0.00      0.00       132
         alkane       0.81      0.99      0.89      2121
         alkene       0.00      0.00      0.00       375
        alcohol       0.00      0.00      0.00       667
          arene       0.79      0.81      0.80      1670
          amine       0.00      0.00      0.00       798
         ketone       0.00      0.00      0.00       260
          ether       0.70      0.09      0.15       749
          imine       0.00      0.00      0.00        35
    sulfonamide       0.00      0.00      0.00        41
    acyl_halide       0.00      0.00      0.00        11
      phosphate       0.00      0.00      0.00        49
       aldehyde       0.00      0.00      0.00        56

In [10]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.91, 0.05)

rows = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    rows.append({
        "threshold": t,
        "micro_f1": f1_score(y_test, y_pred_t, average="micro", zero_division=0),
        "macro_f1": f1_score(y_test, y_pred_t, average="macro", zero_division=0),
        "samples_f1": f1_score(y_test, y_pred_t, average="samples", zero_division=0),
    })

import pandas as pd

threshold_df = pd.DataFrame(rows)
threshold_df

,threshold,micro_f1,macro_f1,samples_f1
0,0.10,0.545842,0.216224,0.529084
1,0.15,0.591683,0.208400,0.572771
2,0.20,0.609467,0.195849,0.594510
3,0.25,0.626253,0.184187,0.612653
4,0.30,0.630051,0.171848,0.617080
5,0.35,0.621685,0.152498,0.608808
6,0.40,0.600031,0.112221,0.591214
7,0.45,0.589533,0.097467,0.583272
8,0.50,0.579636,0.092078,0.574084
9,0.55,0.565027,0.088262,0.560816


In [11]:
threshold_df.sort_values("macro_f1", ascending=False).head(10)

,threshold,micro_f1,macro_f1,samples_f1
0,0.10,0.545842,0.216224,0.529084
1,0.15,0.591683,0.208400,0.572771
2,0.20,0.609467,0.195849,0.594510
3,0.25,0.626253,0.184187,0.612653
4,0.30,0.630051,0.171848,0.617080
5,0.35,0.621685,0.152498,0.608808
6,0.40,0.600031,0.112221,0.591214
7,0.45,0.589533,0.097467,0.583272
8,0.50,0.579636,0.092078,0.574084
9,0.55,0.565027,0.088262,0.560816


In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

LABEL_NAMES = [
    "ester", "carboxylic_acid", "alkane", "alkene", "alcohol",
    "arene", "amine", "ketone", "ether", "imine",
    "sulfonamide", "acyl_halide", "phosphate", "aldehyde", "nitro",
    "enamine", "azo", "sulfonic_acid", "amide", "peroxide"
]

# Train/test split again
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Calculate class weights per label
positive_counts = y_train.sum(axis=0)
negative_counts = len(y_train) - positive_counts

# Avoid division by zero
pos_weights = negative_counts / np.maximum(positive_counts, 1)

# Clip weights so rare labels do not explode too much
pos_weights = np.clip(pos_weights, 1.0, 20.0)

print("Positive class weights:")
for name, weight, count in zip(LABEL_NAMES, pos_weights, positive_counts):
    print(f"{name}: weight={weight:.2f}, positives={int(count)}")


def weighted_binary_crossentropy(pos_weights):
    pos_weights_tf = tf.constant(pos_weights, dtype=tf.float32)

    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        # Prevent log(0)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        loss_pos = -y_true * tf.math.log(y_pred) * pos_weights_tf
        loss_neg = -(1.0 - y_true) * tf.math.log(1.0 - y_pred)

        return tf.reduce_mean(loss_pos + loss_neg)

    return loss_fn

X_train: (10579, 1800, 1)
X_test: (2645, 1800, 1)
y_train: (10579, 20)
y_test: (2645, 20)
Positive class weights:
ester: weight=6.13, positives=1484
carboxylic_acid: weight=18.52, positives=542
alkane: weight=1.00, positives=8424
alkene: weight=6.02, positives=1507
alcohol: weight=3.03, positives=2625
arene: weight=1.00, positives=6562
amine: weight=2.40, positives=3107
ketone: weight=9.53, positives=1005
ether: weight=2.61, positives=2927
imine: weight=20.00, positives=114
sulfonamide: weight=20.00, positives=115
acyl_halide: weight=20.00, positives=34
phosphate: weight=20.00, positives=187
aldehyde: weight=20.00, positives=245
nitro: weight=17.99, positives=557
enamine: weight=20.00, positives=47
azo: weight=20.00, positives=104
sulfonic_acid: weight=20.00, positives=120
amide: weight=14.65, positives=676
peroxide: weight=20.00, positives=12


In [13]:
def build_better_ftir_cnn(input_length, num_labels, pos_weights):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),

        layers.Conv1D(32, kernel_size=9, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64, kernel_size=7, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=5, activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling1D(),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),

        layers.Dense(num_labels, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=weighted_binary_crossentropy(pos_weights),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model


input_length = X_train.shape[1]
num_labels = y_train.shape[1]

model = build_better_ftir_cnn(input_length, num_labels, pos_weights)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 1800, 32)       │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1800, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 900, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 900, 64)        │        14,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 900, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 450, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 450, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 450, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 225, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 225, 128)       │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 225, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,588 (490.58 KB)

 Trainable params: 124,884 (487.83 KB)

 Non-trainable params: 704 (2.75 KB)

In [14]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 107s 378ms/step - auc: 0.8004 - binary_accuracy: 0.7307 - loss: 0.8519 - val_auc: 0.8155 - val_binary_accuracy: 0.7785 - val_loss: 0.8403 - learning_rate: 5.0000e-04
Epoch 2/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 103s 390ms/step - auc: 0.8359 - binary_accuracy: 0.7559 - loss: 0.8168 - val_auc: 0.8398 - val_binary_accuracy: 0.7908 - val_loss: 0.8084 - learning_rate: 5.0000e-04
Epoch 3/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 138s 373ms/step - auc: 0.8450 - binary_accuracy: 0.7619 - loss: 0.8013 - val_auc: 0.8686 - val_binary_accuracy: 0.8093 - val_loss: 0.7876 - learning_rate: 5.0000e-04
Epoch 4/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 140s 367ms/step - auc: 0.8457 - binary_accuracy: 0.7621 - loss: 0.7933 - val_auc: 0.8568 - val_binary_accuracy: 0.7509 - val_loss: 0.8069 - learning_rate: 5.0000e-04
Epoch 5/50
265/265 ━━━━━━━━━━━━━━━━━━━━ 98s 371ms/step - auc: 0.8500 - binary_accuracy: 0.7679 - loss: 0.7874 - val_auc: 0.8590 - val_binary_accuracy: 0.7910 - val_loss: 0.8007

In [15]:
y_prob = model.predict(X_test)

thresholds = np.arange(0.05, 0.91, 0.05)

rows = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)

    rows.append({
        "threshold": round(float(t), 2),
        "micro_f1": f1_score(y_test, y_pred_t, average="micro", zero_division=0),
        "macro_f1": f1_score(y_test, y_pred_t, average="macro", zero_division=0),
        "samples_f1": f1_score(y_test, y_pred_t, average="samples", zero_division=0),
    })

import pandas as pd

threshold_df = pd.DataFrame(rows)

print("Best thresholds by macro F1:")
display(threshold_df.sort_values("macro_f1", ascending=False).head(10))

print("Best thresholds by micro F1:")
display(threshold_df.sort_values("micro_f1", ascending=False).head(10))

print("Best thresholds by samples F1:")
display(threshold_df.sort_values("samples_f1", ascending=False).head(10))

83/83 ━━━━━━━━━━━━━━━━━━━━ 15s 175ms/step
Best thresholds by macro F1:


,threshold,micro_f1,macro_f1,samples_f1
10,0.55,0.598720,0.325985,0.581988
9,0.50,0.580993,0.324171,0.570451
11,0.60,0.606709,0.321278,0.583536
12,0.65,0.609324,0.317224,0.578039
8,0.45,0.552824,0.316114,0.549312
7,0.40,0.525921,0.308009,0.525900
13,0.70,0.588971,0.291066,0.547313
6,0.35,0.500299,0.287752,0.501065
5,0.30,0.474911,0.273243,0.475271
4,0.25,0.448850,0.257311,0.449206


Best thresholds by micro F1:


,threshold,micro_f1,macro_f1,samples_f1
12,0.65,0.609324,0.317224,0.578039
11,0.60,0.606709,0.321278,0.583536
10,0.55,0.598720,0.325985,0.581988
13,0.70,0.588971,0.291066,0.547313
9,0.50,0.580993,0.324171,0.570451
8,0.45,0.552824,0.316114,0.549312
14,0.75,0.540653,0.256533,0.480132
7,0.40,0.525921,0.308009,0.525900
6,0.35,0.500299,0.287752,0.501065
5,0.30,0.474911,0.273243,0.475271


Best thresholds by samples F1:


,threshold,micro_f1,macro_f1,samples_f1
11,0.60,0.606709,0.321278,0.583536
10,0.55,0.598720,0.325985,0.581988
12,0.65,0.609324,0.317224,0.578039
9,0.50,0.580993,0.324171,0.570451
8,0.45,0.552824,0.316114,0.549312
13,0.70,0.588971,0.291066,0.547313
7,0.40,0.525921,0.308009,0.525900
6,0.35,0.500299,0.287752,0.501065
14,0.75,0.540653,0.256533,0.480132
5,0.30,0.474911,0.273243,0.475271


In [16]:
BEST_THRESHOLD = 0.55

y_pred = (y_prob >= BEST_THRESHOLD).astype(int)

micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
samples_f1 = f1_score(y_test, y_pred, average="samples", zero_division=0)

print("Final Improved CNN Evaluation")
print("Threshold:", BEST_THRESHOLD)
print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)
print("Samples F1:", samples_f1)

print(classification_report(
    y_test,
    y_pred,
    target_names=LABEL_NAMES,
    zero_division=0
))

Final Improved CNN Evaluation
Threshold: 0.55
Micro F1: 0.59872
Macro F1: 0.325985132030552
Samples F1: 0.581987790432136
                 precision    recall  f1-score   support

          ester       0.37      0.69      0.48       412
carboxylic_acid       0.12      0.77      0.20       132
         alkane       0.82      0.98      0.89      2121
         alkene       0.27      0.42      0.33       375
        alcohol       0.44      0.67      0.53       667
          arene       0.81      0.77      0.79      1670
          amine       0.51      0.55      0.53       798
         ketone       0.24      0.51      0.32       260
          ether       0.49      0.56      0.52       749
          imine       0.57      0.49      0.52        35
    sulfonamide       0.44      0.20      0.27        41
    acyl_halide       0.00      0.00      0.00        11
      phosphate       0.12      0.39      0.18        49
       aldehyde       0.30      0.50      0.38        56
          nitro       

In [17]:
model.save("weighted_ftir_cnn_threshold_055.keras")

In [18]:
import pandas as pd

cnn_final_summary = pd.DataFrame([{
    "model": "Weighted 1D CNN",
    "threshold": BEST_THRESHOLD,
    "micro_f1": micro_f1,
    "macro_f1": macro_f1,
    "samples_f1": samples_f1,
}])

cnn_final_summary.to_csv("weighted_cnn_summary.csv", index=False)
cnn_final_summary

,model,threshold,micro_f1,macro_f1,samples_f1
0,Weighted 1D CNN,0.55,0.59872,0.325985,0.581988


In [19]:
from google.colab import files

files.download("weighted_ftir_cnn_threshold_055.keras")
files.download("weighted_cnn_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>